Before Filtering

In [1]:
pip install jsonlines

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Run this interactively in notebooks/exploration.ipynb
from pathlib import Path
from itertools import islice
import jsonlines
import pandas as pd

# Resolve data path for either cwd = project root or cwd = notebooks/
if Path.cwd().name == "notebooks":
    data_path = Path("../data/raw/klc_part_001_of_008.jsonl")
else:
    data_path = Path("data/raw/klc_part_001_of_008.jsonl")

# Peek at first 20 records
with jsonlines.open(data_path) as f:
    samples = list(islice(f, 20))

# See all field names
df_sample = pd.DataFrame(samples)
print("Columns:", df_sample.columns.tolist())

# Show category values only if that column exists
if "category" in df_sample.columns:
    print(df_sample["category"].value_counts())
else:
    print("No 'category' column in this file.")

# Show a preview using whichever common columns are present
preview_cols = [
    c for c in ["year", "source", "doc_type", "language", "script", "text_composition", "text"]
    if c in df_sample.columns
]
print(df_sample[preview_cols].head(10))

Columns: ['id', 'url', 'source', 'corpus', 'doc_type', 'copyright_status', 'year', 'language', 'script', 'text_composition', 'text', 'content', 'translation', 'metadata', 'text_analytics']
No 'category' column in this file.
   year                                            source       doc_type  \
0  None  Institute for the Translation of Korean Classics  literary_work   
1  None  Institute for the Translation of Korean Classics  literary_work   
2  None  Institute for the Translation of Korean Classics  literary_work   
3  None  Institute for the Translation of Korean Classics  literary_work   
4  None  Institute for the Translation of Korean Classics  literary_work   
5  None  Institute for the Translation of Korean Classics  literary_work   
6  None  Institute for the Translation of Korean Classics  literary_work   
7  None  Institute for the Translation of Korean Classics  literary_work   
8  None  Institute for the Translation of Korean Classics  literary_work   
9  None  Institu

In [6]:
# Deeper schema inspection requested
import json
from pprint import pprint

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

def inspect_cell_value(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "None", None, None
    if isinstance(v, dict):
        return "dict", list(v.keys()), v
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                parsed = json.loads(s)
                if isinstance(parsed, dict):
                    return "JSON str (dict)", list(parsed.keys()), parsed
                return f"JSON str ({type(parsed).__name__})", None, parsed
            except Exception:
                return "plain str", None, v
        return "plain str", None, v
    return type(v).__name__, None, v

# 1) Unpack nested columns and print full raw values
print("\n=== 1) Full raw values for nested-like columns ===")
for col in ["metadata", "text_analytics", "content"]:
    print(f"\n--- Column: {col} ---")
    val = df_sample.iloc[0][col] if col in df_sample.columns else None
    vtype, keys, parsed_or_raw = inspect_cell_value(val)
    print("Type:", vtype)
    if keys is not None:
        print("Keys:", keys)
    print("Full value:")
    pprint(parsed_or_raw, width=200, sort_dicts=False)

# 2) Get all unique values / counts
print("\n=== 2) value_counts for selected columns ===")
for col in ["corpus", "doc_type", "language", "script", "text_composition"]:
    print(f"\n--- {col} ---")
    if col in df_sample.columns:
        print(df_sample[col].value_counts(dropna=False))
    else:
        print("Column not found")

# 3) Spot-check one likely poem (short text < 100 chars), print full row
print("\n=== 3) Spot-check a short text row (<100 chars) ===")
if "text" in df_sample.columns:
    text_len = df_sample["text"].astype(str).str.len()
    short_rows = df_sample[text_len < 100]
    if not short_rows.empty:
        row = short_rows.iloc[0]
        print("Chosen row index:", row.name, "| text length:", len(str(row["text"])))
        print("Full row values:")
        for c in df_sample.columns:
            print(f"\n[{c}]")
            pprint(row[c], width=200, sort_dicts=False)
    else:
        print("No rows with text length < 100 found in current sample.")
else:
    print("Column 'text' not found")

# 4) Explicit type/key checks for metadata + text_analytics
print("\n=== 4) Type/key check for metadata and text_analytics ===")
for col in ["metadata", "text_analytics"]:
    print(f"\n--- {col} ---")
    val = df_sample.iloc[0][col] if col in df_sample.columns else None
    vtype, keys, parsed_or_raw = inspect_cell_value(val)
    print("Is value:", vtype)
    if keys is not None:
        print("Dict keys:", keys)
    else:
        print("Dict keys: N/A")
    print("One full example value:")
    pprint(parsed_or_raw, width=200, sort_dicts=False)


=== 1) Full raw values for nested-like columns ===

--- Column: metadata ---
Type: dict
Keys: ['book_author', 'book_id', 'book_name', 'book_pub_year', 'book_size', 'elem_copyright', 'elem_copyright_ko', 'elem_copyright_punc', 'elem_dci', 'elem_dci_ko', 'elem_dci_punc', 'elem_id', 'elem_id_ko', 'elem_id_punc', 'elem_url', 'elem_url_ko', 'elem_url_punc', 'page_path']
Full value:
{'book_author': '최치원(崔致遠)',
 'book_id': 'ITKC_MO_0001A',
 'book_name': '계원필경집(桂苑筆耕集)',
 'book_pub_year': '1834',
 'book_size': '1집',
 'elem_copyright': 'ⓒ 한국고전번역원 | 영인표점 한국문집총간 | 1990',
 'elem_copyright_ko': 'ⓒ 한국고전번역원 | 이상현 (역) | 2009',
 'elem_copyright_punc': 'ⓒ 한국고전번역원 | 2018',
 'elem_dci': 'ITKC_MO_0001A_0010_010_0010_2003_A001_XML',
 'elem_dci_ko': 'ITKC_BT_0001A_0010_000_0010_2011_001_XML',
 'elem_dci_punc': 'ITKC_MP_0001A_0010_000_0010_2021_001_XML',
 'elem_id': 'ITKC_MO_0001A_0010_010_0010',
 'elem_id_ko': 'ITKC_BT_0001A_0010_000_0010',
 'elem_id_punc': 'ITKC_MP_0001A_0010_000_0010',
 'elem_url': 'https:

In [7]:
# Compact summary of the inspection
import json

def type_and_keys(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "None", None
    if isinstance(v, dict):
        return "dict", list(v.keys())
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                parsed = json.loads(s)
                if isinstance(parsed, dict):
                    return "JSON str(dict)", list(parsed.keys())
                return f"JSON str({type(parsed).__name__})", None
            except Exception:
                return "plain str", None
        return "plain str", None
    return type(v).__name__, None

print("Nested columns type/key summary:")
for c in ["metadata", "text_analytics", "content"]:
    t, k = type_and_keys(df_sample.iloc[0][c] if c in df_sample.columns else None)
    print(f"- {c}: {t}; keys={k}")

print("\nUnique counts summary:")
for c in ["corpus", "doc_type", "language", "script", "text_composition"]:
    if c in df_sample.columns:
        vc = df_sample[c].value_counts(dropna=False)
        print(f"- {c}: {vc.to_dict()}")
    else:
        print(f"- {c}: missing")

if "text" in df_sample.columns:
    lengths = df_sample["text"].astype(str).str.len()
    short_rows = df_sample[lengths < 100]
    if not short_rows.empty:
        r = short_rows.iloc[0]
        print("\nShort-text spot-check:")
        print(f"- row index: {r.name}")
        print(f"- text length: {len(str(r['text']))}")
        print(f"- text: {r['text']}")
    else:
        print("\nShort-text spot-check: none under 100 chars in current sample")

Nested columns type/key summary:
- metadata: dict; keys=['book_author', 'book_id', 'book_name', 'book_pub_year', 'book_size', 'elem_copyright', 'elem_copyright_ko', 'elem_copyright_punc', 'elem_dci', 'elem_dci_ko', 'elem_dci_punc', 'elem_id', 'elem_id_ko', 'elem_id_punc', 'elem_url', 'elem_url_ko', 'elem_url_punc', 'page_path']
- text_analytics: dict; keys=['text_length', 'content_body_length', 'content_body_punc_length', 'content_body_xml_punc_length', 'content_elem_title_length', 'content_elem_title_ko_length', 'content_elem_title_punc_length', 'content_title_length', 'translation_body_ko_length', 'translation_footnote_ko_length']
- content: dict; keys=['body', 'body_punc', 'body_xml_punc', 'elem_title', 'elem_title_ko', 'elem_title_punc', 'title']

Unique counts summary:
- corpus: {'Korean Literary Collections': 20}
- doc_type: {'literary_work': 20}
- language: {'Hanmun': 20}
- script: {'Hanja': 20}
- text_composition: {'{body}': 20}

Short-text spot-check: none under 100 chars in c

In [8]:
# Find a short-text candidate by scanning more records
from itertools import islice

scan_n = 5000
with jsonlines.open(data_path) as f:
    scan_samples = list(islice(f, scan_n))

df_scan = pd.DataFrame(scan_samples)
if "text" in df_scan.columns:
    scan_lengths = df_scan["text"].astype(str).str.len()
    short_scan = df_scan[scan_lengths < 100]
    print(f"Scanned records: {len(df_scan)}")
    print(f"Short-text records (<100 chars): {len(short_scan)}")
    if not short_scan.empty:
        srow = short_scan.iloc[0]
        print("\nOne short-text full row:")
        for c in df_scan.columns:
            print(f"\n[{c}]")
            pprint(srow[c], width=200, sort_dicts=False)
    else:
        print("No short-text rows found in the first", scan_n, "records.")
else:
    print("Column 'text' not found while scanning.")

Scanned records: 5000
Short-text records (<100 chars): 2621

One short-text full row:

[id]
'klc:ITKC_MO_0001A_0110_010_0100'

[url]
'https://db.itkc.or.kr/dir/item?itemId=MO#/dir/node?dataId=ITKC_MO_0001A_0110_010_0100&viewSync=KP&viewSync2=TR'

[source]
'Institute for the Translation of Korean Classics'

[corpus]
'Korean Literary Collections'

[doc_type]
'literary_work'

[copyright_status]
'Public Domain'

[year]
None

[language]
'Hanmun'

[script]
'Hanja'

[text_composition]
'{body}'

[text]
'伏以禮慶履長。傳標視朔。夷夏契混同之運。乾坤叶交泰之期。伏惟司空相公浙水流恩。吳山變俗。旣睹趙衰之日。永洽物情。願親傅說之星。早環帝座。末由拜賀。但切禱祈云云。'

[content]
{'body': '伏以禮慶履長。傳標視朔。夷夏契混同之運。乾坤叶交泰之期。伏惟司空相公浙水流恩。吳山變俗。旣睹趙衰之日。永洽物情。願親傅說之星。早環帝座。末由拜賀。但切禱祈云云。',
 'body_punc': '伏以禮慶履長，傳標視朔，夷、夏契混同之運，乾坤叶交泰之期。伏惟司空相公浙水流恩，吳山變俗，旣睹趙衰之日，永洽物情；願親傅說之星，早環帝座。末由拜賀，但切禱祈云云。',
 'body_xml_punc': '伏以禮慶履長，傳標視朔，夷、夏契混同之運，乾坤叶交泰之期。伏惟司空相公<ner_5fb636>浙水</ner_5fb636>流恩，<ner_5fb636>吳山</ner_5fb636>變俗，旣睹<ner_5fb636>趙衰</ner_5fb636>之日，永洽物情；願親<ner_5fb636>傅說</ner_5fb636>之星，早環帝座。末由拜賀，但切禱祈云云。',
 'elem_titl

In [10]:
import jsonlines
import glob
from pathlib import Path
from collections import Counter
from tqdm import tqdm

# Resolve raw-data glob regardless of whether cwd is project root or notebooks/
if Path.cwd().name == "notebooks":
    pattern = "../data/raw/klc_part_*.jsonl"
else:
    pattern = "data/raw/klc_part_*.jsonl"

files = sorted(glob.glob(pattern))
print(f"Using pattern: {pattern}")
print(f"Files found: {len(files)}")

genre_counter = Counter()

for fpath in files:
    with jsonlines.open(fpath) as reader:
        for doc in tqdm(reader, desc=Path(fpath).name):
            title = doc.get("content", {}).get("title", "") or ""
            parts = title.split("/")
            genre = parts[-1].strip() if len(parts) > 1 else "NO_SLASH"
            genre_counter[genre] += 1

if not genre_counter:
    print("No genres counted. Check file pattern and title extraction logic.")
else:
    print("\nTop genres by frequency:")
    for genre, count in genre_counter.most_common(60):
        print(f"{count:>8,}  {genre}")

Using pattern: ../data/raw/klc_part_*.jsonl
Files found: 8


klc_part_001_of_008.jsonl: 96126it [00:01, 49668.20it/s]
klc_part_002_of_008.jsonl: 86570it [00:01, 46550.42it/s]
klc_part_003_of_008.jsonl: 76187it [00:01, 44924.33it/s]
klc_part_004_of_008.jsonl: 67601it [00:01, 42738.99it/s]
klc_part_005_of_008.jsonl: 62373it [00:01, 40974.13it/s]
klc_part_006_of_008.jsonl: 92104it [00:01, 51905.99it/s]
klc_part_007_of_008.jsonl: 96973it [00:01, 55471.12it/s]
klc_part_008_of_008.jsonl: 74471it [00:01, 53752.56it/s]


Top genres by frequency:
 186,424  詩
  79,601  書
  20,961  [詩]
  13,136  七言絶句
  13,065  祭文
  11,239  雜著
  10,749  附錄
   9,978  七言律詩
   8,669  序
   7,930  NO_SLASH
   7,830  記
   7,613  詩○七言絶句
   6,745  五言律詩
   5,083  疏箚
   4,696  疏
   4,112  跋
   4,047  墓碣銘
   3,578  詩○七言律詩
   3,507  行狀
   3,417  五言絶句
   3,261  詩類
   3,096  七言律
   2,935  墓表
   2,539  墓誌銘
   2,341  詩○五言絶句
   2,268  詩○五言律詩
   2,268  詩 西河任相元公輔著
   2,214  題跋
   2,161  五言古詩
   2,116  五言律
   1,959  䟽
   1,940  祝文
   1,937  詩○七言律
   1,794  律詩
   1,573  墓誌
   1,520  七言古詩
   1,490  啓辭
   1,476  銘
   1,463  䟽箚
   1,386  啓
   1,190  詩 蔡彭胤仲耆甫著
   1,173  賦
   1,063  墓碣
   1,062  書牘
   1,009  詩○五言古詩
     995  詩○五言律
     938  七言四韻
     896  [書]
     876  文
     842  說
     841  議
     832  詩○咸安郡
     809  箚
     806  [文]
     742  哀辭
     736  上樑文
     731  五言排律
     727  傳
     692  燕行錄
     684  古律詩


In [13]:
import pandas as pd

df = pd.read_csv("/Users/marvinchen/Desktop/EAS 407 - Minor/eas minor/data/processed/klc_features.csv")

# Find a qijue with exactly 8 lines (2 quatrains) and inspect raw text
sample = df[(df["form_label"] == "qijue") & (df["num_lines"] == 8)].iloc[0]
print("Title:", sample["title_full"])
print("Author:", sample["author"])
print("Raw text:")
print(sample["text"])
print("\nNum lines detected:", sample["num_lines"])

Title: 梅湖遺稿 / 七言絶句
Author: 진화(陳澕)
Raw text:
酒痕微微點玉顋。暗香搖蕩隔林人。紅杏紫桃無遠韻。一枝都占上園春。又風輕不用臙脂雪。月冷潛驚玉露香。別院曉寒煙淡淡。數枝和睡靚新粧。

Num lines detected: 8


In [15]:
# In your notebook
df = pd.read_csv("/Users/marvinchen/Desktop/EAS 407 - Minor/eas minor/data/processed/klc_features.csv")
general = df[df["form_label"] == "general"].copy()

# What line lengths do unlabeled poems cluster at?
print(general["modal_line_length"].value_counts().head(10))
print(general["num_lines"].value_counts().head(10))

# What fraction happen to conform to a lüshi template even without a label?
def infer_form(row):
    n = row["num_lines"]
    l = row["avg_line_length"]
    if n % 4 == 0 and abs(l - 7.0) <= 0.5: return "qijue-like"
    if n % 4 == 0 and abs(l - 5.0) <= 0.5: return "wujue-like"
    if n % 8 == 0 and abs(l - 7.0) <= 0.5: return "qilu-like"
    if n % 8 == 0 and abs(l - 5.0) <= 0.5: return "wulu-like"
    return "non-conformant"

general["inferred"] = general.apply(infer_form, axis=1)
print(general["inferred"].value_counts())
print(f"\n% conforming to some lüshi template: {(general['inferred'] != 'non-conformant').mean()*100:.1f}%")

modal_line_length
7     138140
5      70501
4       1080
6        643
8         94
3         72
9         57
1         14
10        13
2         11
Name: count, dtype: int64
num_lines
8     102187
4      43810
16     14229
12      6788
24      4689
20      3538
10      2874
32      2133
9       2071
40      1873
Name: count, dtype: int64
inferred
qijue-like        121028
wujue-like         58954
non-conformant     30664
Name: count, dtype: int64

% conforming to some lüshi template: 85.4%
